In [ ]:
"""
High-Themes-Only Miner for Android Instrumentation CI Challenges (Remote-Only)

INPUT:
  C:\GitHub\Temp_Data\html_url.csv   (must contain column: html_url with https://github.com/owner/repo)

OUTPUTS (in C:\GitHub\Temp_Data):
  - challenge_counts_weighted_high_only.csv
  - challenge_counts_by_provider_weighted_high_only.csv
  - challenge_examples_high_only.csv
  - challenge_run_audit_high_only.csv
"""

import os, re, io, zipfile, json
from pathlib import Path
import pandas as pd
import numpy as np

# ----------------- CONFIG -----------------
INPUT_CSV        = r"C:\GitHub\Temp_Data\html_url.csv"
OUT_DIR          = r"C:\GitHub\Temp_Data"
TOKENS_ENV_PATH  = Path(r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env")

ENABLE_GITHUB_FETCH = True
MAX_RUNS_PER_REPO   = 30     # increase if you want a deeper sample
REQUEST_TIMEOUT     = 30

# ----------------- THEMES -----------------
THEMES = {
  # High (only these will be counted)
  "EMULATOR_BOOT_TIMEOUT": ["device offline", "boot completed timeout", "emulator: ERROR", "emulator: Panic", "Adb connection refused"],
  "HW_ACCEL/KVM_MISSING": ["/dev/kvm permission denied", "KVM is required", "accel not installed", "HAXM is not installed", "Hypervisor.framework is required"],
  "MISSING_EMULATOR_BIN": ["emulator: not found", "No emulator installed"],
  "ADB_ISSUE": ["ADB server didn't ACK", "ADB server version mismatch", "error: device offline", "more than one device/emulator"],
  "X11/HEADLESS": ["xvfb-run: error", "Cannot open display", "DISPLAY not set"],
  "VENDOR_LAB_QUOTA": ["Quota exceeded", "Billing account not configured", "insufficient tokens for"],

  # (Defined but ignored for counting—left here for completeness)
  "SDK_LICENSES": ["You have not accepted the license agreements", "licenses not accepted", "SDK licenses not accepted"],
  "IMAGE_NOT_FOUND": [r"Package .* was not found", r"failed to find target with hash string", r"system-images;android-\d+.*not found"],
  "JAVA/GRADLE/AGP_VERSION_DRIFT": ["Minimum supported Gradle is", "This version of the Android Gradle plugin requires", "Unsupported Java", "Kotlin version .* is not compatible"],
  "ANDROIDX_TEST_MISMATCH": ["Could not resolve androidx.test", "Duplicate class .* found in modules", "conflict with dependency 'androidx.test'"],
  "NETWORK_FLAKE": ["Connection reset by peer", "TLS handshake timeout", "temporary failure in name resolution", "network is unreachable"],
  "SECRETS_PERMS": ["Permission denied", "No such file or directory .*keystore", "secrets.* not set", "Missing .* environment variable"],
  "GENERIC_TIMEOUT/CANCEL": ["job timed out", "The operation was canceled", "timeout exceeded"],
  "TEST_FLAKY_SIGNAL": [r"Flaky", r"retrying test", r"java\.lang\.AssertionError", r"org\.junit\.ComparisonFailure"],
}

HIGH_THEMES = {
    "EMULATOR_BOOT_TIMEOUT","HW_ACCEL/KVM_MISSING","MISSING_EMULATOR_BIN",
    "ADB_ISSUE","X11/HEADLESS","VENDOR_LAB_QUOTA"
}

# Simple YAML hints (not used for gating, just context in audit)
YAML_HINTS = [
    "reactivecircus/android-emulator-runner",
    "sdkmanager",
    "avdmanager",
    "emulator -avd",
    "gcloud firebase test android run"
]

# ----------------- HELPERS -----------------
def warn(msg): print(f"[WARN] {msg}")
def info(msg): print(f"[INFO] {msg}")

def load_tokens_from_env_file(path: Path) -> dict:
    tokens = {}
    try:
        if path and path.exists():
            for raw in path.read_text(encoding="utf-8", errors="ignore").splitlines():
                line = raw.strip()
                if not line or line.startswith("#") or line.startswith(";") or "=" not in line:
                    continue
                k, v = line.split("=", 1)
                tokens[k.strip()] = v.strip().strip('"').strip("'")
    except Exception as e:
        warn(f"Failed reading tokens env: {e}")
    return tokens

def resolve_github_token() -> str:
    file_tokens = load_tokens_from_env_file(TOKENS_ENV_PATH)
    tok = file_tokens.get("GITHUB_TOKEN_1", "") or \
          os.environ.get("GITHUB_TOKEN_1", "") or \
          os.environ.get("GITHUB_TOKEN", "")
    return tok.strip()

GITHUB_TOKEN = resolve_github_token()
if ENABLE_GITHUB_FETCH:
    if not GITHUB_TOKEN:
        warn("No GITHUB_TOKEN resolved; will try unauthenticated (public-only).")
    else:
        info(f"Loaded token length={len(GITHUB_TOKEN)}; prefix={GITHUB_TOKEN[:6]!r}")

def gh_headers(allow_auth=True):
    base = {"Accept": "application/vnd.github+json", "User-Agent": "challenge-miner/high-only/1.0"}
    if allow_auth and GITHUB_TOKEN:
        base["Authorization"] = f"Bearer {GITHUB_TOKEN}"
    return base

def gh_get(url, allow_auth=True):
    import requests
    resp = requests.get(url, headers=gh_headers(allow_auth), timeout=REQUEST_TIMEOUT)
    # attach diagnostics for audit
    diag = {
        "status": resp.status_code,
        "message": None,
        "rate_remaining": resp.headers.get("X-RateLimit-Remaining"),
        "rate_reset": resp.headers.get("X-RateLimit-Reset"),
    }
    try:
        j = resp.json()
        if isinstance(j, dict) and "message" in j:
            diag["message"] = j["message"]
    except Exception:
        pass
    resp._diag = diag
    return resp

def owner_repo_from_url(url: str):
    try:
        u = (url or "").strip().strip("/")
        if "github.com/" in u:
            tail = u.split("github.com/", 1)[1].strip("/")
            parts = tail.split("/")
            if len(parts) >= 2:
                return parts[0].lower(), parts[1].lower()
    except Exception:
        pass
    return None, None

# ----------------- FETCH (Logs & Workflows) -----------------
def fetch_gha_logs(owner, repo, max_runs=MAX_RUNS_PER_REPO):
    """Paginated; tries auth first, falls back to unauth for public repos."""
    out, page, err = [], 1, None
    per_page, got = min(max_runs, 100), 0
    while got < max_runs:
        runs = gh_get(f"https://api.github.com/repos/{owner}/{repo}/actions/runs?per_page={per_page}&page={page}", allow_auth=True)
        if runs.status_code == 401:
            warn(f"401 with token for {owner}/{repo} — retrying without auth.")
            runs = gh_get(f"https://api.github.com/repos/{owner}/{repo}/actions/runs?per_page={per_page}&page={page}", allow_auth=False)
        if runs.status_code != 200:
            err = runs._diag
            break
        wr = runs.json().get("workflow_runs", [])
        if not wr:
            break
        for run in wr:
            if got >= max_runs: break
            logzip = gh_get(run["logs_url"], allow_auth=True)
            if logzip.status_code == 401:
                logzip = gh_get(run["logs_url"], allow_auth=False)
            if logzip.status_code != 200:
                err = logzip._diag
                continue
            zf = zipfile.ZipFile(io.BytesIO(logzip.content))
            texts = []
            for name in zf.namelist():
                try:
                    with zf.open(name) as f:
                        texts.append(f.read().decode("utf-8", errors="ignore"))
                except Exception:
                    continue
            out.append((run["id"], "\n".join(texts)))
            got += 1
        page += 1
    return out, err

def fetch_workflow_yamls(owner, repo):
    """Remote workflow YAMLs for context (not gating results)."""
    yamls, err = [], None
    for allow_auth in (True, False):
        r = gh_get(f"https://api.github.com/repos/{owner}/{repo}/contents/.github/workflows", allow_auth=allow_auth)
        if r.status_code in (404, 403):
            err = r._diag; break
        if r.status_code == 200:
            for item in r.json():
                if item.get("type") == "file" and item.get("name", "").lower().endswith((".yml",".yaml")):
                    raw_url = item.get("download_url")
                    if not raw_url: continue
                    fr = gh_get(raw_url, allow_auth=allow_auth)
                    if fr.status_code == 200:
                        yamls.append(fr.text)
            break
    return "\n".join(yamls), err

# ----------------- SCANNING -----------------
PAT_MAP = {k: [re.compile(p, re.I) for p in v] for k, v in THEMES.items()}

def scan_text_for_themes(text: str):
    """Return (found_themes, hits) scanning line-by-line to avoid multiline misses."""
    found, hits = set(), []
    if not text: return [], []
    for theme, regs in PAT_MAP.items():
        for line in text.splitlines():
            for rx in regs:
                if rx.search(line):
                    found.add(theme)
                    hits.append((theme, line[:500]))
                    break
            if theme in found: break
    return list(found), hits

def scan_yaml_hints(text: str):
    L = (text or "").lower()
    return [h for h in YAML_HINTS if h in L]

# ----------------- MAIN -----------------
def main():
    out = Path(OUT_DIR); out.mkdir(parents=True, exist_ok=True)

    # Load URLs
    urls = pd.read_csv(INPUT_CSV)
    if "html_url" not in urls.columns:
        raise ValueError("Input CSV must include a column 'html_url'.")
    urls["html_url"] = urls["html_url"].astype(str)

    # Prepare repo list
    repos = []
    for url in urls["html_url"]:
        o, r = owner_repo_from_url(url)
        if o and r:
            repos.append(f"{o}/{r}")
    if not repos:
        raise ValueError("No valid GitHub repo URLs found.")

    rows, examples, audit = [], [], []

    for full in repos:
        provider, stratum, w = "github_actions", "sample", 1.0
        o, r = full.split("/", 1)

        # 1) Logs
        logs, runs_err = fetch_gha_logs(o, r, max_runs=MAX_RUNS_PER_REPO)

        # 2) Workflows (context only)
        yaml_text, wf_err = fetch_workflow_yamls(o, r)
        yaml_hints = scan_yaml_hints(yaml_text) if yaml_text else []

        # 3) Scan logs, then keep only HIGH themes
        seen_themes = set()
        kept_examples = []
        if logs:
            for run_id, txt in logs:
                fthemes, fhits = scan_text_for_themes(txt)
                for th in fthemes:
                    if th in HIGH_THEMES:
                        seen_themes.add(th)
                # collect examples for HIGH only (max 3 per repo)
                for th, snip in fhits:
                    if th in HIGH_THEMES:
                        kept_examples.append((th, run_id, snip))

        # Trim to up to 3 examples per repo
        ex_kept = 0
        for th, run_id, snip in kept_examples:
            if ex_kept >= 3: break
            examples.append({
                "full_name": full, "provider": provider, "stratum": stratum,
                "theme": th, "run_id": run_id, "snippet": (snip or "").strip()[:500]
            })
            ex_kept += 1

        rows.append({
            "full_name": full,
            "provider": provider,
            "stratum": stratum,
            "weight": w,
            "themes": ";".join(sorted(seen_themes)) if seen_themes else "",
            "yaml_hints": ";".join(yaml_hints) if yaml_hints else "",
            "log_runs_scanned": len(logs),
            "yaml_bytes": len(yaml_text),
            "runs_err": json.dumps(runs_err) if runs_err else "",
            "wf_err": json.dumps(wf_err) if wf_err else "",
        })

        audit.append({
            "full_name": full,
            "log_runs_scanned": len(logs),
            "yaml_bytes": len(yaml_text),
            "yaml_hints": ";".join(yaml_hints) if yaml_hints else "",
            "themes_high_only": ";".join(sorted(seen_themes)) if seen_themes else "",
            "rate_remaining": (runs_err or {}).get("rate_remaining") if runs_err else None,
        })

    df = pd.DataFrame(rows)

    # Explode high-only themes for counts
    df_exp = df.copy()
    df_exp["themes"] = df_exp["themes"].fillna("")
    df_exp = df_exp[df_exp["themes"] != ""]
    if not df_exp.empty:
        df_exp = df_exp.assign(theme=df_exp["themes"].str.split(";")).explode("theme")
    else:
        df_exp = pd.DataFrame(columns=["full_name","provider","stratum","weight","theme"])

    # Weighted counts (weights are 1.0 here)
    if not df_exp.empty:
        grp = df_exp.groupby("theme").agg(
            weighted_count=("weight", lambda s: float(np.nansum(s))),
            repos=("full_name","nunique"),
        ).reset_index()
        total_w = float(np.nansum(df["weight"].values))
        grp["share"] = np.where(total_w>0, grp["weighted_count"]/total_w, np.nan)
        grp = grp.sort_values("weighted_count", ascending=False)
    else:
        grp = pd.DataFrame(columns=["theme","weighted_count","repos","share"])

    # By provider (still useful if you ever add other providers)
    if not df_exp.empty:
        byprov = df_exp.groupby(["provider","theme"]).agg(
            weighted_count=("weight", lambda s: float(np.nansum(s))),
            repos=("full_name","nunique"),
        ).reset_index()
        totals = byprov.groupby("provider")["weighted_count"].sum().rename("total_w")
        byprov = byprov.merge(totals, on="provider", how="left")
        byprov["share_in_provider"] = np.where(byprov["total_w"]>0, byprov["weighted_count"]/byprov["total_w"], np.nan)
        byprov = byprov.drop(columns=["total_w"]).sort_values(["provider","weighted_count"], ascending=[True, False])
    else:
        byprov = pd.DataFrame(columns=["provider","theme","weighted_count","repos","share_in_provider"])

    # Save
    out = Path(OUT_DIR); out.mkdir(parents=True, exist_ok=True)
    grp.to_csv(out/"challenge_counts_weighted_high_only.csv", index=False, encoding="utf-8-sig")
    byprov.to_csv(out/"challenge_counts_by_provider_weighted_high_only.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(examples).to_csv(out/"challenge_examples_high_only.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(audit).to_csv(out/"challenge_run_audit_high_only.csv", index=False, encoding="utf-8-sig")

    print(f"Saved -> {out/'challenge_counts_weighted_high_only.csv'}")
    print(f"Saved -> {out/'challenge_counts_by_provider_weighted_high_only.csv'}")
    print(f"Saved -> {out/'challenge_examples_high_only.csv'}")
    print(f"Saved -> {out/'challenge_run_audit_high_only.csv'}")

if __name__ == "__main__":
    main()


[INFO] Loaded token length=40; prefix='ghp_AU'
[WARN] Runs list HTTP 404 for 8cayqpvkio/android-2048-compose-material3
[WARN] Runs list HTTP 404 for afermon/mypass-app
[WARN] Runs list HTTP 404 for alaory/wallme-wallpaper
[WARN] Runs list HTTP 404 for alvarenga-dev/marketplace-list
[WARN] Runs list HTTP 404 for aperii/android
[WARN] Runs list HTTP 404 for avalax/fitbuddy
[WARN] Runs list HTTP 403 for balzack/databag
[WARN] Runs list HTTP 403 for banana-boat/huaxiaoyu-fe
[WARN] Runs list HTTP 403 for bandev/buddhaquotes
[WARN] Runs list HTTP 403 for barnhill/bible
[WARN] Runs list HTTP 403 for bartuzen/qbitcontroller
[WARN] Runs list HTTP 403 for baseballcardtracker/bbct-android
[WARN] Runs list HTTP 403 for battle-buddy/battlebuddy-android
[WARN] Runs list HTTP 403 for bauerj/paperless_app
[WARN] Runs list HTTP 403 for bcgov/bcvax-android
[WARN] Runs list HTTP 403 for bcgov/fbp-go
[WARN] Runs list HTTP 403 for bcgov/secure-image-app
[WARN] Runs list HTTP 403 for bclynch/edmflare
[WARN]